# Anarkali: build the web-expert dataset (CPU)

Streams Common Crawl, builds `web-expert-v1` with `scripts/build_web_expert.py`, and uploads it to
`RUN_NAME/web-expert-v1/` on HF. No GPU needed. Kaggle: Accelerator None, Internet on, secret
HF_TOKEN. Colab: any runtime; the token is asked for if no secret is set.


In [ ]:
RUN_NAME = "run3"
HF_REPO = "toufiqqureshi651/anarkali-web"
REPO_URL = "https://github.com/ToufiqQureshi/anarkali.git"
BRANCH = "main"
CRAWL = "CC-MAIN-2026-39"
WARCS = 12          # normal pages, ~30-40k HTML pages each
ERROR_WARCS = 3     # 404 / 403 / 500 pages

In [ ]:
import os, subprocess, sys, json
from pathlib import Path

WORK = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path("/content")
REPO = WORK / "anarkali"
OUT = WORK / "web-expert-v1"
LOG = WORK / "data-build-log.txt"

def sh(cmd, cwd=None):
    """Run a command, streaming its output to the notebook and LOG; raise if it fails."""
    print(f"$ {cmd}", flush=True)
    proc = subprocess.Popen(cmd, shell=True, cwd=cwd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                            text=True, errors="replace", bufsize=1, env={**os.environ, "PYTHONUNBUFFERED": "1"})
    with open(LOG, "a", encoding="utf-8") as log:
        for line in proc.stdout:
            print(line, end="", flush=True)
            log.write(line)
    if proc.wait():
        raise subprocess.CalledProcessError(proc.returncode, cmd)

if not REPO.exists():
    sh(f"git clone --depth 1 --branch {BRANCH} {REPO_URL} {REPO}")
sh("git log --oneline -1", cwd=REPO)

def _hf_token():
    token = os.environ.get("HF_TOKEN")
    for getter in ("kaggle", "colab"):
        if token:
            break
        try:
            if getter == "kaggle":
                from kaggle_secrets import UserSecretsClient
                token = UserSecretsClient().get_secret("HF_TOKEN")
            else:
                from google.colab import userdata
                token = userdata.get("HF_TOKEN")
        except Exception:
            pass
    if not token:
        from getpass import getpass   # no secrets panel (e.g. VS Code): ask; the token is not saved
        token = getpass("Hugging Face token (hf_...): ").strip()
    return token

from huggingface_hub import HfApi
hf = HfApi(token=_hf_token())
print("Hugging Face user:", hf.whoami()["name"], "| CPUs:", os.cpu_count())

In [ ]:
try:
    sh(f"python scripts/build_web_expert.py --crawl {CRAWL} --warcs {WARCS} --error-warcs {ERROR_WARCS} "
       f"--workers {max(2, os.cpu_count())} --output {OUT}", cwd=REPO)
finally:
    hf.upload_file(repo_id=HF_REPO, path_or_fileobj=str(LOG), path_in_repo=f"{RUN_NAME}/data-build-log.txt")
hf.upload_folder(repo_id=HF_REPO, folder_path=str(OUT), path_in_repo=f"{RUN_NAME}/web-expert-v1")
manifest = json.loads((OUT / "manifest.json").read_text())
print("uploaded", f"{HF_REPO}/{RUN_NAME}/web-expert-v1", "| revision", manifest["revision"])
for split, c in manifest["split_counts"].items():
    print(f"{split:12s} {c['decision_cases']:7d} rows from {c['source_groups']:6d} sites")